# Pandas: Bedingen, Gruppieren und Bayes
### Unterrichtsmaterial | Julia Brutskaya

---
**Kontext:** Wahrscheinlichkeitsrechnung, Tag 2 – Datenwald-Akademie<br>
**Bibliotheken:** `pandas`, `numpy`

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="130"> <img src="png%20Dateien/etta_elster.png" alt="Etta" width="130"> <img src="png%20Dateien/nils_eichhoernchen.png" alt="Nils" width="130"> <img src="png%20Dateien/mats_maulwurf.png" alt="Mats Maulwurf" width="130">

Neulich ging es um bedingte Wahrscheinlichkeit, die Vierfeldertafel und Bayes. Der wichtigste Satz für heute mit pandas:

> **Bedingen ist Filtern.** Erst filtern, dann zählen. Die Bedingung steht im Filter, die Frage in `value_counts` oder `mean`.

Das ist nicht nur Wahrscheinlichkeitsrechnung, sondern Alltag in der Datenanalyse: Retourenquote **pro Zahlart**, Ausfallquote **pro Server**, Spam-Anteil **je nach Stichwort**.

| Block | Inhalt | Neu in Pandas |
|---|---|---|
| A | Bedingen ist Filtern | `groupby` auf True/False-Spalten, `idxmax` |
| B | Gruppieren und zählen | `groupby` + `value_counts(normalize=True)`, MultiIndex, `stack`/`unstack` |
| C | Die Vierfeldertafel selbst normieren | `sum(axis=…)`, `div(…, axis=…)`, `index.repeat` |
| D | Unabhängigkeit prüfen | die erwartete Tafel mit `np.outer` |
| E | Bayes mit 100 000 Dateien | `np.where` |
| F | Ein Mini-Spamfilter und weitere Übungen | `.str.contains` + `groupby` |

---
# Block A – Bedingen ist Filtern

## 1 – Die Kurs-Umfrage laden

In [2]:
import pandas as pd
import numpy as np

zeilen = [
    # teilnehmer_id, kurszeit, getraenk, tshirt, haustiere, anfahrt_min, lernstunden, punkte
    (101, "Vormittag", "Kaffee", "M", 0, 25, 3, 50), (102, "Abend", "Tee", "S", 1, 35, 4, 58),
    (103, "Vormittag", "Mate", "L", 0, 18, 4, 63), (104, "Vormittag", "Kaffee", "L", 2, 42, 9, 81),
    (105, "Abend", "Wasser", "M", 0, 30, 8, 84), (106, "Vormittag", "Tee", "M", 1, 28, 5, 60),
    (107, "Abend", "Kaffee", "XL", 0, 55, 9, 86), (108, "Vormittag", "Kaffee", "S", 3, 12, 7, 79),
    (109, "Vormittag", "Mate", "M", 1, 38, 3, 57), (110, "Abend", "Tee", "M", 0, 115, 12, 66),
    (111, "Vormittag", "Kaffee", "L", 0, 32, 10, 88), (112, "Abend", "Mate", "L", 2, 45, 1, 47),
    (113, "Vormittag", "Wasser", "S", 1, 22, 6, 72), (114, "Vormittag", "Kaffee", "M", 0, 35, 5, 62),
    (115, "Abend", "Tee", "L", 1, 48, 11, 94), (116, "Vormittag", "Tee", "M", 0, 30, 8, 76),
    (117, "Abend", "Kaffee", "M", 3, 65, 2, 52), (118, "Vormittag", "Mate", "XL", 1, 40, 6, 69),
    (119, "Abend", "Wasser", "S", 0, 52, 7, 70), (120, "Vormittag", "Kaffee", "L", 2, 25, 6, 61),
]
spalten = ["teilnehmer_id", "kurszeit", "getraenk", "tshirt", "haustiere",
           "anfahrt_min", "lernstunden", "punkte"]
umfrage = pd.DataFrame(zeilen, columns=spalten)
umfrage.head()

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min,lernstunden,punkte
0,101,Vormittag,Kaffee,M,0,25,3,50
1,102,Abend,Tee,S,1,35,4,58
2,103,Vormittag,Mate,L,0,18,4,63
3,104,Vormittag,Kaffee,L,2,42,9,81
4,105,Abend,Wasser,M,0,30,8,84


## 2 – Erst filtern, dann zählen

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="120">

> **Herr Bauchgefühl:** „40 % trinken Kaffee. Also trinkt auch der Abendkurs zu 40 % Kaffee."

P(Kaffee | Abend) heißt: Wir schauen **nur** auf den Abendkurs. Der Abendkurs ist der neue Ergebnisraum, also der Nenner.

In [3]:
abendkurs = umfrage[umfrage["kurszeit"] == "Abend"]
print("Personen im Abendkurs:", len(abendkurs))
print()
print(abendkurs["getraenk"].value_counts(normalize=True))
print()

# dasselbe mit query und einer Maske
print("P(Kaffee | Abend):", umfrage.query('kurszeit == "Abend"')["getraenk"].eq("Kaffee").mean())
print("P(Kaffee) ohne Bedingung:", (umfrage["getraenk"] == "Kaffee").mean())

Personen im Abendkurs: 8

getraenk
Tee       0.375
Wasser    0.250
Kaffee    0.250
Mate      0.125
Name: proportion, dtype: float64

P(Kaffee | Abend): 0.25
P(Kaffee) ohne Bedingung: 0.4


Im Abendkurs trinken nur 25 % Kaffee, nicht 40 %. Die 40 % gelten für alle 20 Personen.

## 3 – `groupby` auf eine True/False-Spalte

Statt für jede Gruppe einzeln zu filtern, rechnet `groupby` alle Gruppen auf einmal. Die Frage wird eine True/False-Spalte, ihr Mittelwert ist der Anteil.

In [4]:
umfrage["kaffee"] = umfrage["getraenk"] == "Kaffee"
print(umfrage.groupby("kurszeit")["kaffee"].mean())

kurszeit
Abend        0.25
Vormittag    0.50
Name: kaffee, dtype: float64


> **Merksatz:** Wonach gruppiert wird, ist die **Bedingung**. Sie steht hinter dem Strich: `groupby("kurszeit")` liefert P(… | Kurszeit).

Andersherum: Wie groß ist P(Vormittag | Getränk)? Und bei welchem Getränk ist der Anteil am größten?

In [5]:
umfrage["vormittag"] = umfrage["kurszeit"] == "Vormittag"
anteil_vormittag = umfrage.groupby("getraenk")["vormittag"].mean()
print(anteil_vormittag.round(3))
print()
print("größter Anteil:", anteil_vormittag.idxmax())

getraenk
Kaffee    0.750
Mate      0.750
Tee       0.400
Wasser    0.333
Name: vormittag, dtype: float64

größter Anteil: Kaffee


`idxmax()` liefert den Index (hier das Getränk) mit dem größten Wert. **Achtung:** Kaffee und Mate haben beide 0,75. Bei Gleichstand nennt `idxmax` nur den ersten. Kaffee ist also nicht „der Sieger", sondern gleichauf mit Mate.

> **Im Job:** Genau so rechnest du die Conversion-Rate pro Kanal oder die Retourenquote pro Zahlart: eine True/False-Spalte, `groupby` nach der Gruppe, `mean()`.

### Übung 1

Übung 1 vom Vormittag. Berechne mit Filtern oder `groupby`:

- **a)** P(Tee | Abend) und P(Abend | Tee)
- **b)** P(Mate | Vormittag) und P(Vormittag | Mate)
- **c)** Warum sind die beiden Werte in a) verschieden?

In [ ]:
# a) Bedingung = Filter, Frage = Mittelwert der Maske
tee = umfrage["getraenk"] == "Tee"
abend = umfrage["kurszeit"] == "Abend"
print("P(Tee | Abend)", umfrage[abend]["getraenk"].eq("Tee").mean()) # 3 von 8
print("P(Abend | Tee)", umfrage[tee]["kurszeit"].eq("Abend").mean()) # 3 von 5


P(Tee | Abend) 0.375
P(Abend | Tee) 0.6


In [49]:
# b)
mate = umfrage["getraenk"] == "Mate"
vormittag = umfrage["kurszeit"] == "Vormittag"
print("P(Mate | Vormittag)", umfrage[vormittag]["getraenk"].eq("Mate").mean())
print("P(Vormittag | Mate)", umfrage[mate]["kurszeit"].eq("Vormittag").mean()) 


P(Mate | Vormittag) 0.25
P(Vormittag | Mate) 0.75


In [ ]:
# c)



---
# Block B – Gruppieren und zählen

## 4 – `groupby` + `value_counts(normalize=True)`

`value_counts(normalize=True)` nach einem `groupby` normiert **innerhalb jeder Gruppe**. Heraus kommt eine Series mit zwei Index-Ebenen, einem **MultiIndex**.

In [9]:
b = umfrage.groupby("kurszeit")["getraenk"].value_counts(normalize=True)
print(b.round(3))
print()
print("Index-Ebenen:", b.index.names)

kurszeit   getraenk
Abend      Tee         0.375
           Wasser      0.250
           Kaffee      0.250
           Mate        0.125
Vormittag  Kaffee      0.500
           Mate        0.250
           Tee         0.167
           Wasser      0.083
Name: proportion, dtype: float64

Index-Ebenen: ['kurszeit', 'getraenk']


Einen einzelnen Wert holst du mit `.loc` und einem **Tupel** aus beiden Ebenen:

In [10]:
print("P(Tee | Abend):", b.loc[("Abend", "Tee")])

P(Tee | Abend): 0.375


Bei einer Series geht auch `b.loc["Abend", "Tee"]` ohne Klammern. Bei einem DataFrame heißt das Komma aber „Zeile, Spalte“. Schreib deshalb immer das Tupel mit runden Klammern.

## 5 – `unstack` und `stack`

Die lange Liste ist schwer zu lesen. `unstack()` dreht die **letzte** Index-Ebene in die Spalten, und es entsteht eine Tabelle. `unstack(level=0)` dreht die **erste** Ebene. `stack()` macht daraus wieder die lange Form.

In [50]:
print(b.unstack().round(3))
print()
print(b.unstack(level=0).round(3))
print()
print(b.unstack().stack().head(8).round(3))

getraenk   Kaffee   Mate    Tee  Wasser
kurszeit                               
Abend        0.25  0.125  0.375   0.250
Vormittag    0.50  0.250  0.167   0.083

kurszeit  Abend  Vormittag
getraenk                  
Kaffee    0.250      0.500
Mate      0.125      0.250
Tee       0.375      0.167
Wasser    0.250      0.083

kurszeit   getraenk
Abend      Kaffee      0.250
           Mate        0.125
           Tee         0.375
           Wasser      0.250
Vormittag  Kaffee      0.500
           Mate        0.250
           Tee         0.167
           Wasser      0.083
dtype: float64


<img src="png%20Dateien/etta_elster.png" alt="Etta" width="110">

**Etta:** „Zeilenprozente sind bedingte Wahrscheinlichkeiten. Jede Zeile ist eine eigene kleine Welt und ergibt zusammen 1."

Das ist dasselbe wie `pd.crosstab(..., normalize="index")` aus Statistik 1. Zwei Wege führen zum Ziel: `crosstab` liefert die Tabelle, `groupby` die lange Liste.

In [54]:
pd.crosstab(umfrage["kurszeit"], umfrage["getraenk"], normalize="index").round(3)

getraenk,Kaffee,Mate,Tee,Wasser
kurszeit,,,,
Abend,0.25,0.125,0.375,0.250
Vormittag,0.50,0.250,0.167,0.083


> **Achtung:** `value_counts` nach `groupby` sortiert innerhalb jeder Gruppe nach Häufigkeit, nicht alphabetisch. Für eine Tabelle immer `unstack()` benutzen.

### Übung 2

Übung 1 noch einmal, diesmal mit `groupby` und `value_counts`.

- **a)** Erstelle `nach_getraenk = umfrage.groupby("getraenk")["kurszeit"].value_counts(normalize=True)`. Lies P(Abend | Tee) mit `.loc` ab.
- **b)** Stelle beide Richtungen als Tabelle dar: P(Getränk | Kurszeit) und P(Kurszeit | Getränk).
- **c)** In welcher der beiden Tabellen ergeben die **Zeilen** 1, in welcher die **Spalten**?

In [55]:
# a)
nach_getraenk = umfrage.groupby("getraenk")["kurszeit"].value_counts(normalize=True)
print("P( Abend | Tee)", nach_getraenk.loc[("Tee", "Abend")])

P( Abend | Tee) 0.6


In [61]:
# b)
print(b.round(3))
print()
print("P(Getränk | Kurszeit)")
print(b.unstack().round(3))
print()
print("P(Kurszeit | Getränk)")
print(nach_getraenk.unstack().round(3))

kurszeit   getraenk
Abend      Tee         0.375
           Wasser      0.250
           Kaffee      0.250
           Mate        0.125
Vormittag  Kaffee      0.500
           Mate        0.250
           Tee         0.167
           Wasser      0.083
Name: proportion, dtype: float64

P(Getränk | Kurszeit)
getraenk   Kaffee   Mate    Tee  Wasser
kurszeit                               
Abend        0.25  0.125  0.375   0.250
Vormittag    0.50  0.250  0.167   0.083

P(Kurszeit | Getränk)
kurszeit  Abend  Vormittag
getraenk                  
Kaffee    0.250      0.750
Mate      0.250      0.750
Tee       0.600      0.400
Wasser    0.667      0.333


In [ ]:
# c) transponiert
print(nach_getraenk.unstack().sum(axis=1))

getraenk
Kaffee    1.0
Mate      1.0
Tee       1.0
Wasser    1.0
dtype: float64


---
# Block C – Die Vierfeldertafel selbst normieren

## 6 – Feld durch Rand

> **Herr Bauchgefühl:** „Wer abends kommt, hat seltener ein Haustier. Der Abendkurs macht haustierfrei!"

`normalize="index"` nimmt dir die Arbeit ab. Heute machen wir es einmal selbst, damit „Feld durch Rand" aus dem Skript sichtbar wird.

In [16]:
tab = pd.crosstab(umfrage["kurszeit"], umfrage["haustiere"] > 0)
tab.columns = ["kein Haustier", "Haustier"]      # statt False / True
print(tab)
print()
print("Zeilenränder:", tab.sum(axis=1).tolist())
print("Spaltenränder:", tab.sum(axis=0).tolist())

           kein Haustier  Haustier
kurszeit                          
Abend                  4         4
Vormittag              5         7

Zeilenränder: [8, 12]
Spaltenränder: [9, 11]


<img src="png%20Dateien/nils_eichhoernchen.png" alt="Nils" width="110">

**Nils:** „Ich prüfe immer die Ecke unten rechts: Zeilenränder und Spaltenränder müssen beide 20 ergeben."

`sum(axis=1)` summiert jede Zeile, `sum(axis=0)` jede Spalte. Jetzt teilen wir jedes Feld durch seinen Rand mit `div`:

In [17]:
# Jede Zeile durch ihre Zeilensumme: P(Haustier | Kurszeit)
print(tab.div(tab.sum(axis=1), axis=0).round(3))
print()
# Jede Spalte durch ihre Spaltensumme: P(Kurszeit | Haustier)
print(tab.div(tab.sum(axis=0), axis=1).round(3))
print()
# Durch die Gesamtsumme: Wahrscheinlichkeits-Tafel
print((tab / tab.values.sum()).round(2))

           kein Haustier  Haustier
kurszeit                          
Abend              0.500     0.500
Vormittag          0.417     0.583

           kein Haustier  Haustier
kurszeit                          
Abend              0.444     0.364
Vormittag          0.556     0.636

           kein Haustier  Haustier
kurszeit                          
Abend               0.20      0.20
Vormittag           0.25      0.35


Im Abendkurs haben 50 % ein Haustier, im Vormittagskurs 58,3 %. Das ist ein kleiner Unterschied bei nur 8 und 12 Personen.

> **Stolperstein:** Ohne `axis=0` versucht Pandas, die Zeilensummen an den **Spalten** auszurichten. Weil die Namen nicht passen, kommt nur `NaN` heraus:

In [18]:
tab.div(tab.sum(axis=1))

,Abend,Haustier,Vormittag,kein Haustier
kurszeit,,,,
Abend,NaN,NaN,NaN,NaN
Vormittag,NaN,NaN,NaN,NaN


## 7 – Von Anzahlen zurück zu Rohdaten: `index.repeat`

Aufgaben liefern oft nur eine Häufigkeitstabelle. `index.repeat` wiederholt jede Zeile so oft, wie in der Spalte `anzahl` steht. So entstehen Rohdaten, eine Zeile pro Bestellung.

In [66]:
haeuf = pd.DataFrame({
    "zahlart": ["PayPal", "PayPal", "Rechnung", "Rechnung"],
    "retoure": [True, False, True, False],
    "anzahl": [20, 220, 40, 120],
})
print(haeuf)
print()

bestellungen = haeuf.loc[haeuf.index.repeat(haeuf["anzahl"]), ["zahlart", "retoure"]].reset_index(drop=True)
print("Zeilen:", len(bestellungen))
bestellungen.sample(6, random_state=42)

    zahlart  retoure  anzahl
0    PayPal     True      20
1    PayPal    False     220
2  Rechnung     True      40
3  Rechnung    False     120

Zeilen: 400


,zahlart,retoure
209,PayPal,False
280,Rechnung,False
33,PayPal,False
210,PayPal,False
93,PayPal,False
84,PayPal,False


### Übung 3

Übung 3 vom Vormittag: die 400 Bestellungen aus Abschnitt 7.

- **a)** Berechne die Retourenquote pro Zahlart mit `groupby`.
- **b)** Berechne P(Rechnung | Retoure).
- **c)** Baue die Vierfeldertafel mit `crosstab` und berechne die Spaltenprozente von Hand mit `div`.

In [68]:
# a) Bedíngung = Zahlart
print(bestellungen.groupby("zahlart")["retoure"].mean().round(3))


zahlart
PayPal      0.083
Rechnung    0.250
Name: retoure, dtype: float64


In [72]:
# b) Bedingung = Retoure, Frage = Zahlart
nach_retoure = bestellungen.groupby("retoure")["zahlart"].value_counts(normalize=True)
print("P(Rechnung | Retoure)", round(nach_retoure.loc[(True, "Rechnung")], 3))


P(Rechnung | Retoure) 0.667


In [ ]:
# c) Spaltenprozente = je Spalte / ihre Summe
tafel = pd.crosstab(bestellungen["zahlart"], bestellungen["retoure"])
tafel.columns = ["keine Retoure", "Retoure"]
print(tafel)
print()
print(tafel.div(tafel.sum(axis=0), axis=1).round(3))                    # 2 von 3 Retouren kommen von Rechnung

          keine Retoure  Retoure
zahlart                         
PayPal              220       20
Rechnung            120       40

          keine Retoure  Retoure
zahlart                         
PayPal            0.647    0.333
Rechnung          0.353    0.667


---
# Block D – Unabhängigkeit prüfen

## 8 – Die erwartete Tafel mit `np.outer`

Wären Kurszeit und Haustier **unabhängig**, müsste jedes Feld gleich Zeilenrand · Spaltenrand sein. `np.outer` multipliziert jeden Zeilenrand mit jedem Spaltenrand.

In [23]:
p = pd.crosstab(umfrage["kurszeit"], umfrage["haustiere"] > 0, normalize="all")
p.columns = ["kein Haustier", "Haustier"]

zeilenrand = p.sum(axis=1)
spaltenrand = p.sum(axis=0)
erwartet = pd.DataFrame(np.outer(zeilenrand, spaltenrand), index=p.index, columns=p.columns)

print("beobachtet:")
print(p.round(2))
print()
print("erwartet bei Unabhängigkeit:")
print(erwartet.round(2))
print()
print("Abweichung:")
print((p - erwartet).round(2))

beobachtet:
           kein Haustier  Haustier
kurszeit                          
Abend               0.20      0.20
Vormittag           0.25      0.35

erwartet bei Unabhängigkeit:
           kein Haustier  Haustier
kurszeit                          
Abend               0.18      0.22
Vormittag           0.27      0.33

Abweichung:
           kein Haustier  Haustier
kurszeit                          
Abend               0.02     -0.02
Vormittag          -0.02      0.02


<img src="png%20Dateien/herr_bauchgefuehl_verdutzt.png" alt="Herr Bauchgefühl stutzt" width="120">

Jedes Feld weicht nur um 0,02 ab. Bei 20 Personen ist das „fast unabhängig". Der Abendkurs macht also nicht haustierfrei.

<img src="png%20Dateien/der_ausreisser.png" alt="Der Ausreißer" width="100">

**Der Ausreißer:** „Bei 20 Leuten verschiebt ein einziger Kater die Quote."

Ob eine Abweichung Zufall ist, klärt erst ein statistischer Test (Kapitel 5). Ausblick: Genau diese erwartete Tafel ist der erste Schritt des Chi²-Tests.

> **Im Job:** Hängt die Kündigungsquote vom Tarif ab? Hängt die Fehlerquote von der Schicht ab? Beobachtet gegen erwartet ist der erste Blick.

### Übung 4

Übung 9 vom Vormittag: Hängt die Retoure an der Zahlart? Arbeite mit `bestellungen`.

- **a)** Berechne die Wahrscheinlichkeits-Tafel (`normalize="all"`) und die erwartete Tafel bei Unabhängigkeit.
- **b)** Vergleiche P(Retoure) mit P(Retoure | PayPal) und P(Retoure | Rechnung).
- **c)** Sind Zahlart und Retoure unabhängig?

In [83]:
# a) normalize = geteilt duch, all = Gesamtzahl -> P(A und B), index = Zeilensumme -> P(Spalte | Zeile) oder P(Passwort | Telefon), 
# columns = Spaltensumme -> P (Zeile | Spalte) oder (P(Telefon | Passwort))
p_ret = pd.crosstab(bestellungen["zahlart"], bestellungen["retoure"], normalize="all")
p_ret.columns = ["keine Retoure", "Retoure"]
erwartet_ret = pd.DataFrame(np.outer(p_ret.sum(axis=1), p_ret.sum(axis=0)), index=p_ret.index, columns=p_ret.columns)
erwartet_ret_2 = pd.DataFrame(np.outer(p_ret.sum(axis=0), p_ret.sum(axis=1)), index=p_ret.columns, columns=p_ret.index)
print(p_ret.round(3))
print()
print(erwartet_ret)
print()
print(erwartet_ret_2.round(3))


          keine Retoure  Retoure
zahlart                         
PayPal             0.55     0.05
Rechnung           0.30     0.10

          keine Retoure  Retoure
zahlart                         
PayPal             0.51     0.09
Rechnung           0.34     0.06

zahlart        PayPal  Rechnung
keine Retoure    0.51      0.34
Retoure          0.09      0.06


In [ ]:
# b)
print("P(Retoure)", bestellungen["retoure"].mean())                  # ohne Bedingung
print(bestellungen.groupby("zahlart")["retoure"].mean().round(3))    # mit Bedingung



P(Retoure) 0.15
zahlart
PayPal      0.083
Rechnung    0.250
Name: retoure, dtype: float64


In [26]:
# c)


---
# Block E – Bayes mit 100 000 Dateien

## 9 – Der Virenscanner

> **Herr Bauchgefühl:** „Der Scanner erkennt 99 % aller Viren. Wenn er Alarm schlägt, ist die Datei also zu 99 % infiziert."

Die Angaben aus dem Skript:

- 1 % aller Dateien sind infiziert (Basisrate).
- Bei einer infizierten Datei schlägt der Scanner in 99 % der Fälle Alarm (Trefferquote).
- Bei einer sauberen Datei schlägt er in 5 % der Fälle Alarm (Fehlalarm).

Wir erzeugen 100 000 Dateien. `np.where(bedingung, wenn_ja, wenn_nein)` wählt **pro Zeile**: Ist die Datei infiziert, gilt die Trefferquote, sonst die Fehlalarmquote.

In [27]:
rng = np.random.default_rng(42)
n = 100_000

dateien = pd.DataFrame({"infiziert": rng.random(n) < 0.01})
dateien["alarm"] = np.where(dateien["infiziert"], rng.random(n) < 0.99, rng.random(n) < 0.05)

pd.crosstab(dateien["infiziert"], dateien["alarm"], margins=True)

alarm,False,True,All
infiziert,,,
False,94089,4864,98953
True,14,1033,1047
All,94103,5897,100000


Jetzt in **beide Richtungen** gruppieren:

In [28]:
print("P(Alarm | Virus):")
print(dateien.groupby("infiziert")["alarm"].mean().round(3))
print()
print("P(Virus | Alarm):")
print(dateien.groupby("alarm")["infiziert"].mean().round(3))

P(Alarm | Virus):
infiziert
False    0.049
True     0.987
Name: alarm, dtype: float64

P(Virus | Alarm):
alarm
False    0.000
True     0.175
Name: infiziert, dtype: float64


<img src="png%20Dateien/mats_maulwurf_huegel.png" alt="Mats Maulwurf" width="140">

**Mats Maulwurf:** „Ich bin die Basisrate. Unter den Alarmen stecken fast fünfmal so viele saubere wie infizierte Dateien."

- `groupby("infiziert")["alarm"]` ist P(Alarm | Virus), etwa 0,99. Das kennt Herr Bauchgefühl.
- `groupby("alarm")["infiziert"]` ist P(Virus | Alarm), etwa 0,17. Das will er eigentlich wissen.

Die Formel im Skript liefert 0,167, die Simulation einen Wert in der Nähe. Das ist kein Fehler, sondern Zufall wie an Tag 1.

> **Stolperstein:** `groupby("alarm")["infiziert"]` ist nicht dasselbe wie `groupby("infiziert")["alarm"]`. Genau das ist die Falle „Bedingung vertauschen" aus dem Skript.

> **Herr Bauchgefühl:** „90 % aller Hacker benutzen Linux. Wer Linux benutzt, ist also verdächtig.“

Dieselbe Falle: 90 % ist P(Linux | Hacker). Für einen Verdacht bräuchte man P(Hacker | Linux). In Pandas wären das zwei verschiedene Zeilen:

- `groupby("hacker")["linux"].mean()` → P(Linux | Hacker)
- `groupby("linux")["hacker"].mean()` → P(Hacker | Linux)

Fast alle Linux-Nutzer sind ganz normale Admins, Entwickler und Studierende. Der zweite Wert ist deshalb winzig.

## 10 – Der zweite Scan

Ein zweiter, unabhängiger Scanner prüft dieselben Dateien. Wie groß ist P(Virus), wenn **beide** Alarm schlagen?

In [29]:
dateien["alarm2"] = np.where(dateien["infiziert"], rng.random(n) < 0.99, rng.random(n) < 0.05)
beide = dateien["alarm"] & dateien["alarm2"]

print("Dateien mit zwei Alarmen:", beide.sum())
print("P(Virus | zwei Alarme):  ", round(dateien.loc[beide, "infiziert"].mean(), 3))

Dateien mit zwei Alarmen: 1295
P(Virus | zwei Alarme):   0.791


Mit zwei Alarmen steigt die Wahrscheinlichkeit auf rund 80 %. Der erste Alarm hat die Basisrate von 1 % auf etwa 17 % gehoben, der zweite startet schon von dort.

### Übung 5

Übung 11 vom Vormittag. Ein anderer Scanner: Basisrate 0,1 %, Trefferquote 99 %, Fehlalarm 1 %.

- **a)** Simuliere **1 000 000** Dateien mit `rng = np.random.default_rng(11)`.
- **b)** Berechne P(Virus | Alarm). Vergleiche mit der Formel 99 / 1098 ≈ 0,090.
- **c)** Warum brauchst du hier 1 000 000 Dateien statt 100 000?

In [30]:
# a)


In [31]:
# b)


In [32]:
# c)


---
# Block F – Ein Mini-Spamfilter und weitere Übungen

- **Abschnitt 11:** der Mini-Spamfilter
- **Übung 6:** Spamfilter simulieren (zu Block E)
- **Übung 7:** Fehlgeschlagene Logins (zu Block E)
- **Übung 8:** Ein eigenes Wort im Spamfilter (zu Abschnitt 11)
- **Übung 9:** Eine Tafel aus Wahrscheinlichkeiten (zu Block C und D)

## 11 – Ein Mini-Spamfilter

Ein Postfach mit 13 **erfundenen** Betreffzeilen, jede ist als Spam oder kein Spam markiert. Enthält der Betreff das Wort „Gewinn", wie wahrscheinlich ist es dann Spam?

In [33]:
mails = pd.DataFrame({
    "betreff": ["Gewinn! Sie haben gewonnen", "Protokoll Teammeeting", "Ihr Gewinn wartet",
                "Rechnung Oktober", "Gratis Gewinn sichern", "Re: Gewinn- und Verlustrechnung Q3",
                "Passwort zurücksetzen", "Jetzt gratis testen", "Kursplan nächste Woche",
                "Gewinnspiel: iPhone gratis", "Server-Wartung am Samstag", "Einladung Sommerfest",
                "Kredit sofort ohne Schufa"],
    "spam": [True, False, True, False, True, False, False, True, False, True, False, False, True],
})
mails["gewinn"] = mails["betreff"].str.contains("gewinn", case=False)

print("Basisrate Spam:", round(mails["spam"].mean(), 3))
print()
print("P(Spam | 'Gewinn'):")
print(mails.groupby("gewinn")["spam"].mean().round(3))
print()
print("P('Gewinn' | Spam):")
print(mails.groupby("spam")["gewinn"].mean().round(3))

Basisrate Spam: 0.462

P(Spam | 'Gewinn'):
gewinn
False    0.25
True     0.80
Name: spam, dtype: float64

P('Gewinn' | Spam):
spam
False    0.143
True     0.667
Name: gewinn, dtype: float64


In [34]:
pd.crosstab(mails["gewinn"], mails["spam"], margins=True)

spam,False,True,All
gewinn,,,
False,6,2,8
True,1,4,5
All,7,6,13


- 80 % der Mails mit „Gewinn" sind Spam. Das ist P(Spam | Wort), was der Filter braucht.
- 67 % der Spam-Mails enthalten „Gewinn". Das ist P(Wort | Spam), eine andere Zahl.
- „Gewinn- und Verlustrechnung Q3" ist eine echte Mail mit dem Wort, also ein Fehlalarm.

Die Daten sind klein und erfunden. Ein echter Filter lernt aus Tausenden Mails und kombiniert viele Wörter (Naive Bayes).

### Übung 6 – Spamfilter simulieren (zu Block E)

Übung 10 vom Vormittag: 20 % aller Mails sind Spam. Der Filter erkennt 95 % des Spams und markiert 2 % der echten Mails fälschlich.

- **a)** Simuliere 100 000 Mails mit `rng = np.random.default_rng(10)` und `np.where`.
- **b)** Wie groß ist P(Spam | markiert)? Vergleiche mit 190 / 206 ≈ 0,922.
- **c)** Warum ist der Wert hier so viel höher als beim Virenscanner?

In [35]:
# a)


In [36]:
# b)


In [37]:
# c)


### Übung 7 – Logins im Firmennetz (zu Block E)

Übung 5 vom Vormittag: 70 % aller Logins kommen aus dem Firmennetz, 30 % von außen. Von den Logins aus dem Firmennetz schlagen 2 % fehl, von denen von außen 10 %.

- **a)** Simuliere 100 000 Logins mit `rng = np.random.default_rng(5)`. *Hinweis: erst die Herkunft mit `np.where(rng.random(n) < 0.7, "Firma", "außen")`, dann den Fehlversuch.*
- **b)** Wie wahrscheinlich schlägt ein beliebiger Login fehl? Vergleiche mit 0,044.
- **c)** Erstelle die Vierfeldertafel mit `crosstab` und `margins=True`. Vergleiche mit deiner Tafel für 1 000 Logins vom Vormittag.
- **d)** Ein Login ist fehlgeschlagen. Wie wahrscheinlich kam er von außen? Vergleiche mit 30/44 ≈ 0,682.

In [38]:
# a)


In [39]:
# b)


In [40]:
# c)


In [41]:
# d)

### Übung 8 – Ein eigenes Wort (zu Abschnitt 11)

- **a)** Lege eine Spalte `gratis` an, die prüft, ob „gratis" im Betreff vorkommt.
- **b)** Berechne P(Spam | „gratis") und P(„gratis" | Spam).
- **c)** Welches Wort ist der bessere Spam-Hinweis, „Gewinn" oder „gratis"? Begründe und nenne die Grenze dieser Aussage.

In [42]:
# a)


In [43]:
# b)


In [44]:
# c)


### Übung 9 – Eine Tafel aus Wahrscheinlichkeiten (zu Block C und D)

Übung 4 vom Vormittag: P(A) = 0,3, P(B) = 0,5 und P(A ∩ B) = 0,12. Die ausgefüllte Tafel:

|  | B | nicht B |
|---|---|---|
| A | 0,12 | 0,18 |
| nicht A | 0,38 | 0,32 |

- **a)** Lege die Tafel als DataFrame an und prüfe die Ränder mit `sum(axis=…)`. Ergibt alles zusammen 1?
- **b)** Berechne P(A | B) und P(B | A) mit `div`.
- **c)** Sind A und B unabhängig? Vergleiche P(A ∩ B) mit P(A) · P(B).

In [45]:
# a)

In [46]:
# b)

In [47]:
# c)

---
## 12 – Zusammenfassung

| Aufgabe | Befehl |
|---|---|
| bedingte Wahrscheinlichkeit | erst filtern: `df[bedingung]["spalte"].value_counts(normalize=True)` |
| für alle Gruppen auf einmal | `df.groupby("bedingung")["true_false_spalte"].mean()` |
| größter Wert und sein Name | `idxmax()` (bei Gleichstand nur der erste) |
| bedingte Verteilung je Gruppe | `groupby("bedingung")["frage"].value_counts(normalize=True)` |
| Wert aus dem MultiIndex | `.loc[("ebene1", "ebene2")]` |
| lange Liste ↔ Tabelle | `unstack()`, `unstack(level=0)`, `stack()` |
| Zeilen- und Spaltenränder | `sum(axis=1)`, `sum(axis=0)` |
| Feld durch Rand | `tab.div(tab.sum(axis=1), axis=0)` (Zeilen), `tab.div(tab.sum(axis=0), axis=1)` (Spalten) |
| Anzahlen zu Rohdaten | `df.loc[df.index.repeat(df["anzahl"])]` |
| erwartete Tafel | `np.outer(zeilenrand, spaltenrand)` |
| wenn-dann pro Zeile | `np.where(bedingung, wenn_ja, wenn_nein)` |
| Wort im Text | `.str.contains("wort", case=False)` |

**Merke:** Wonach gruppiert wird, ist die Bedingung. `groupby("alarm")["infiziert"]` und `groupby("infiziert")["alarm"]` beantworten zwei verschiedene Fragen.

---
*Material erstellt von Julia Brutskaya*